# Ch01-04: 이상치 탐지 - 기계학습 기반 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

하이퍼파라미터 민감도 분석

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_n = np.random.multivariate_normal([0,0],[[1,0.5],[0.5,1]],400)
X_o = np.vstack([np.random.uniform(-6,6,(20,2)),np.random.multivariate_normal([5,-4],0.1*np.eye(2),5)])
X = np.vstack([X_n,X_o]); y = np.array([1]*400+[-1]*25)
Xs = StandardScaler().fit_transform(X); c = 25/len(X)

ne = [50,100,200,500]; f1_if = [f1_score(y,IsolationForest(n_estimators=v,contamination=c,random_state=42).fit_predict(Xs),pos_label=-1) for v in ne]
nn = [5,10,20,50,100]; f1_lof = [f1_score(y,LocalOutlierFactor(n_neighbors=v,contamination=c).fit_predict(Xs),pos_label=-1) for v in nn]
gs = ['scale',0.01,0.1,1.0]; f1_svm = [f1_score(y,OneClassSVM(kernel='rbf',gamma=v,nu=c).fit_predict(Xs),pos_label=-1) for v in gs]

fig,axes = plt.subplots(1,3,figsize=(16,4))
axes[0].bar(range(len(ne)),f1_if); axes[0].set_xticks(range(len(ne))); axes[0].set_xticklabels(ne); axes[0].set_title('IF')
axes[1].bar(range(len(nn)),f1_lof); axes[1].set_xticks(range(len(nn))); axes[1].set_xticklabels(nn); axes[1].set_title('LOF')
axes[2].bar(range(len(gs)),f1_svm); axes[2].set_xticks(range(len(gs))); axes[2].set_xticklabels([str(g) for g in gs]); axes[2].set_title('OCSVM')
plt.tight_layout(); plt.show()


**결과 해석**: IF는 n_estimators에 안정적(100+), LOF는 k에 중간 민감, OCSVM은 gamma에 매우 민감하여 교차검증 필수.

---
## 문제 2 풀이

Isolation Forest 직접 구현

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

np.random.seed(42)

class ITreeNode:
    def __init__(self, left=None, right=None, feat=None, val=None, size=0, leaf=False):
        self.left, self.right, self.feat, self.val, self.size, self.leaf = left, right, feat, val, size, leaf

def _c(n):
    if n<=1: return 0
    return 2*(np.log(n-1)+0.5772)-2*(n-1)/n

def build_itree(X, hlim, h=0):
    n,p = X.shape
    if h>=hlim or n<=1: return ITreeNode(size=n, leaf=True)
    f = np.random.randint(0,p); xmin,xmax = X[:,f].min(), X[:,f].max()
    if xmin==xmax: return ITreeNode(size=n, leaf=True)
    sv = np.random.uniform(xmin,xmax); lm = X[:,f]<sv
    return ITreeNode(left=build_itree(X[lm],hlim,h+1), right=build_itree(X[~lm],hlim,h+1), feat=f, val=sv, size=n)

def path_len(x, node, cl=0):
    if node.leaf: return cl+_c(node.size)
    return path_len(x, node.left if x[node.feat]<node.val else node.right, cl+1)

class IForestCustom:
    def __init__(self, n_trees=100, psi=256):
        self.n_trees, self.psi = n_trees, psi
    def fit(self, X):
        n = X.shape[0]; psi = min(self.psi, n); hl = int(np.ceil(np.log2(psi)))
        self.trees = [build_itree(X[np.random.choice(n,psi,replace=False)], hl) for _ in range(self.n_trees)]
        self._cn = _c(psi); return self
    def score_samples(self, X):
        return np.array([2**(-np.mean([path_len(x,t) for t in self.trees])/self._cn) for x in X])

X_n = np.random.multivariate_normal([0,0,0], np.eye(3), 400)
X_o = np.random.uniform(-6,6,(20,3)); X=np.vstack([X_n,X_o]); y=np.array([0]*400+[1]*20)

ifc = IForestCustom(100,256).fit(X); sc = ifc.score_samples(X)
from sklearn.ensemble import IsolationForest as IF
ss = -IF(n_estimators=100,max_samples=256,random_state=42).fit(X).score_samples(X)
print(f"Custom AUC: {roc_auc_score(y,sc):.4f}, sklearn AUC: {roc_auc_score(y,ss):.4f}")


**결과 해석**: 직접 구현이 sklearn과 유사한 AUC 달성. 이상치는 정상보다 짧은 경로=높은 anomaly score.

---
## 문제 3 풀이

LOF 직접 구현 + 밀도 변동 분석

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

np.random.seed(42)

def compute_lof(X, k=20):
    n=X.shape[0]; D=cdist(X,X); np.fill_diagonal(D,np.inf)
    kd = np.sort(D,axis=1)[:,k-1]; kn = np.argsort(D,axis=1)[:,:k]
    rd = np.zeros((n,k))
    for i in range(n):
        for j in range(k): rd[i,j]=max(kd[kn[i,j]], D[i,kn[i,j]])
    lrd = 1.0/(np.mean(rd,axis=1)+1e-10)
    return np.array([np.mean(lrd[kn[i]])/max(lrd[i],1e-10) for i in range(n)])

X_d = np.random.multivariate_normal([0,0],0.1*np.eye(2),200)
X_s = np.random.multivariate_normal([5,5],np.eye(2),50)
X_o = np.array([[3,3],[8,0],[-3,4]]); X=np.vstack([X_d,X_s,X_o]); y=np.array([0]*250+[1]*3)

lof = compute_lof(X,20)
fig,ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(X[:,0],X[:,1],c=lof,cmap='Reds',s=20,vmin=0.8,vmax=3); plt.colorbar(sc,label='LOF')
ax.scatter(X_o[:,0],X_o[:,1],c='blue',marker='*',s=200,label='참 이상치')
ax.set_title('LOF (밀도 변동 데이터)'); ax.legend(); plt.tight_layout(); plt.show()

X_u = np.random.uniform(0,10,(500,2)); lu = compute_lof(X_u,20)
print(f"균일분포 LOF: mean={lu.mean():.4f}, std={lu.std():.4f}")


**결과 해석**: LOF는 지역 밀도 비교로 밀도 변동에도 강건. 균일분포에서 LOF≈1은 이론적 기대와 일치.

---
## 문제 4 풀이

이상치 탐지 앙상블

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import roc_auc_score

np.random.seed(42)

def gen_scenario(sid, n=400):
    rng = np.random.RandomState(sid)
    if sid==0: Xn,Xo = rng.multivariate_normal([0,0],np.eye(2),n), rng.uniform(-6,6,(20,2)); name='산발적'
    elif sid==1: Xn,Xo = rng.multivariate_normal([0,0],np.eye(2),n), rng.multivariate_normal([5,5],0.3*np.eye(2),15); name='클러스터형'
    elif sid==2: Xn = rng.multivariate_normal([0,0],np.eye(2),n); Xo = np.column_stack([rng.normal(0,0.1,20),rng.uniform(-5,5,20)]); name='축형'
    elif sid==3: Xn = np.vstack([rng.multivariate_normal([0,0],np.eye(2),200),rng.multivariate_normal([6,0],np.eye(2),200)]); Xo = rng.uniform(-3,9,(15,2)); name='다중클러스터'
    else: Xn = rng.exponential(2,(n,2)); Xo = rng.uniform(-2,15,(20,2)); name='비대칭'
    X = np.vstack([Xn,Xo]); y = np.array([0]*len(Xn)+[1]*len(Xo)); return X,y,name

results = []
for sid in range(5):
    X,y,name = gen_scenario(sid); c = y.sum()/len(y); Xs = StandardScaler().fit_transform(X)
    iso = IsolationForest(200,contamination=c,random_state=42); iso.fit(Xs); s1 = -iso.score_samples(Xs)
    lof = LocalOutlierFactor(20,contamination=c); lof.fit_predict(Xs); s2 = -lof.negative_outlier_factor_
    svm = OneClassSVM(kernel='rbf',gamma='scale',nu=max(c,0.01)); svm.fit(Xs); s3 = -svm.score_samples(Xs)
    S = MinMaxScaler().fit_transform(np.column_stack([s1,s2,s3]))
    aucs = [roc_auc_score(y,S[:,i]) for i in range(3)]
    w = np.array(aucs)/sum(aucs)
    a_avg = roc_auc_score(y,S.mean(1)); a_max = roc_auc_score(y,S.max(1))
    a_wt = roc_auc_score(y,(S*w).sum(1))
    results.append({'scenario':name,'IF':aucs[0],'LOF':aucs[1],'OCSVM':aucs[2],'Avg':a_avg,'Max':a_max,'Weighted':a_wt})

df_r = pd.DataFrame(results)
print(df_r.to_string(index=False, float_format='%.3f'))


**결과 해석**: AUC 가중 평균 앙상블이 대부분 시나리오에서 안정적. 앙상블은 단일 모델의 '최악의 경우'를 방지한다.

---
## 확장 토론

### ML 이상치 탐지 선택

| 특성 | 추천 |
|------|------|
| 고차원 대규모 | IF |
| 밀도 변동 | LOF |
| 명확한 경계 | OCSVM |
| 강건성 | 앙상블 |